In [1]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM mbdat10", con=connection2)

connection2.close()



In [2]:
base2

,desatetricod,desatetrides,desatetriestregcod
0,1,EMERGENCIA,1
1,2,CONSULTA EXTERNA,1
2,3,CENT.ASIST.DE ADSCRIPCION,1
3,4,ALTA,1
4,6,CENT.ASIST.DE REFERENCIA,1
5,5,PROCEDIMIENTO,0
6,7,C.A.I.,1
7,9,MORTUORIO,1


In [3]:
a=base2[base2.duplicated(subset=["desatetricod"])]
a

,desatetricod,desatetrides,desatetriestregcod


In [4]:
base2.columns

Index(['desatetricod', 'desatetrides', 'desatetriestregcod'], dtype='object')

In [5]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



base2.to_sql(name='tmp_mbdat10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL mbdat10 FALSO CON LO OBTENIDO DEL ORACLE

query_count_before = "SELECT COUNT(*) FROM mbdat10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla mbdat10 antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbdat10 
ALTER COLUMN desatetricod TYPE CHARACTER (1),
ALTER COLUMN desatetrides TYPE CHARACTER (25);


UPDATE mbdat10 
SET 

desatetrides= t.desatetrides


FROM tmp_mbdat10 t 
WHERE mbdat10.desatetricod = t.desatetricod and mbdat10.desatetricod IS NOT NULL and t.desatetricod IS NOT NULL ;

INSERT INTO mbdat10 
(desatetricod, desatetrides) 

SELECT 
desatetricod, desatetrides

FROM tmp_mbdat10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM mbdat10 
    WHERE mbdat10.desatetricod = tmp_mbdat10.desatetricod and mbdat10.desatetricod IS NOT NULL and tmp_mbdat10.desatetricod IS NOT NULL
);


"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_mbdat10;
DELETE FROM mbdat10 WHERE desatetricod ='';
SELECT COUNT(*) FROM mbdat10;
"""


c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla mbdat10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

base2 = pd.read_sql_query(f"SELECT * FROM mbdat10", con=connection3)

connection3.close()


Cantidad de filas en la tabla mbdat10 antes de la inserción: 8
Cantidad de filas en la tabla mbdat10 después de la inserción: 8
La cantidad de filas insertadas fue de: 0


In [6]:
base2.columns

Index(['desatetricod', 'desatetrides'], dtype='object')

In [7]:
#################################################################################################################################################################################################################################################################################


#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


base2.to_sql(name='tmp_mbdat10', con=engine4, if_exists='replace', index=False)



#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS

query_count_before = "SELECT COUNT(*) FROM dim_emedestri"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emedestri antes de la inserción: {cant_antes}")



query="""
ALTER TABLE tmp_mbdat10 
ALTER COLUMN desatetricod TYPE CHARACTER (1),
ALTER COLUMN desatetrides TYPE CHARACTER (25);


INSERT INTO dim_emedestri (cod_des, des_tri) 
SELECT desatetricod, desatetrides

FROM tmp_mbdat10 
WHERE NOT EXISTS (
    SELECT 1 
    FROM dim_emedestri 
    WHERE dim_emedestri.cod_des = tmp_mbdat10.desatetricod
);

DROP TABLE tmp_mbdat10;

SELECT COUNT(*) FROM dim_emedestri;
"""

c1 = text(query)
cursor = connection4.execute(c1)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla dim_emedestri después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

connection4.close()

Cantidad de filas en la tabla dim_emedestri antes de la inserción: 8
Cantidad de filas en la tabla dim_emedestri después de la inserción: 8
La cantidad de filas insertadas fue de: 0
